# Exercise — Encode Governance Obligations in a Data Contract

**Trailhead Provisions** has an ungoverned `orders` contract: it describes columns but carries no
policy. Extend it with a **governance block** so the validator passes both structural schema
validation and semantic governance checks, then explain how a policy change propagates across
producers and downstream consumers. See `INSTRUCTIONS.md`.

In [ ]:
import seed_data
from governance_toolkit import validate_contract, build_lineage

orders_meta = seed_data._build_catalog().table("orders")
print(f"Catalog metadata for 'orders' -> domain={orders_meta.domain!r}, retention={orders_meta.retention!r}")

contract = {
    "name": "orders",
    "domain": "E-commerce Orders",
    "owner": "orders-team@trailhead.example",
    "version": "1.0.0",
    "columns": [
        {"name": "order_id", "type": "integer"},
        {"name": "customer_id", "type": "integer"},
        {"name": "order_ts", "type": "timestamp"},
        {"name": "status", "type": "string"},
        {"name": "amount", "type": "double"},
        {"name": "currency", "type": "string"},
    ],
}
ok, errs = validate_contract(contract)
print("Base contract valid?", ok, "| errors:", errs)   # provided: fails, no governance block

## 1. Add the governance block
Add `pii_tags`, `retention` (look up the `orders` retention period from the catalog metadata printed above or specify an ISO-8601 duration matching `^P\d+[YMD]$`), and `quality_slos` (list of `{dimension, threshold}` where `0 < threshold <= 1`), and mark PII columns with `col["pii"] = True`. Make `validate_contract` pass both JSON Schema and semantic cross-checks.

In [ ]:
# TODO(you): mark PII column(s) with col["pii"] = True and add contract["governance"] = { ... },
# then re-run until validate_contract(contract) passes.
# Note: validate_contract() checks both JSON Schema structure AND semantic rules
# (col["pii"] == True must match governance["pii_tags"], and each SLO threshold must be in (0, 1]).
# contract["governance"] = {"pii_tags": [...], "retention": "...", "quality_slos": [...]}
ok, errs = validate_contract(contract)
print("Extended contract valid?", ok, "| errors:", errs)

In [ ]:
# Provided: inspect the downstream consumers of `orders` before writing your propagation analysis.
lin = build_lineage()
print("Downstream of 'orders':", lin.trace_downstream("orders"))
lin.events()

## 2. Policy-propagation write-up
Replace the cell below with your write-up.

> **Your propagation write-up here.** Address:
>
> - How the change affects **producers** (what the E-commerce Orders pipeline must now enforce).
> - How it affects **downstream consumers** in `trace_downstream("orders")`: specifically `marketing_audience` (which joins `orders.customer_id` with `customer_360`) and `order_carbon` / `quarterly_sustainability` (which read `orders` for CSRD carbon reporting).
> - The **propagation mechanism** (version-bump + publish gate + upstream-first remediation).